# 🎬 ReAgent-V: Multi-Agent Video Understanding on Kaggle (2x GPU T4)

Notebook này được thiết kế để chạy **ReAgent-V (NeurIPS 2025)** trọn gói trên **Kaggle** mà **KHÔNG CẦN add thêm bất kỳ Data Source nào**.

### ⚙️ Hướng dẫn thiết lập Kaggle trước khi chạy:
1. Ở thanh menu bên phải (**Notebook options**):
   - **Accelerator**: Chọn **`GPU T4 x2`**
   - **Internet**: Bật **`Internet on`** (Bắt buộc để tải repo và weights từ Hugging Face)
2. Nhấn **Run All** hoặc chạy từng cell theo thứ tự.

## 📦 Bước 1: Clone Repository & Cài đặt môi trường

In [ ]:
# 1. Clone repository ReAgent-V
!git clone https://github.com/aiming-lab/ReAgent-V.git /kaggle/working/ReAgent-V-repo

# 2. Cài đặt các thư viện cần thiết
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.40.0 accelerate==0.29.3 bitsandbytes
!pip install -q opencv-python decord librosa soundfile easydict networkx yt-dlp
!pip install -q git+https://github.com/LLaVA-VL/LLaVA-NeXT.git

print("✅ Cài đặt môi trường hoàn tất!")

## 🎥 Bước 2: Tải Video mẫu để thử nghiệm
(Bạn có thể đổi link sang video khác hoặc tải video của bạn lên thư mục `/kaggle/working`)

In [ ]:
import os

video_path = "/kaggle/working/sample_video.mp4"

# Tải một đoạn video mẫu ngắn về chú thỏ Big Buck Bunny
if not os.path.exists(video_path):
    print("Đang tải video mẫu...")
    !wget -q -O {video_path} "https://commondatastorage.googleapis.com/gtv-videos-bucket/sample/ForBiggerBlazes.mp4"
    print(f"✅ Đã tải video mẫu tại: {video_path}")
else:
    print(f"Video đã tồn tại: {video_path}")

## 🤖 Bước 3: Cấu hình Model Weights từ Hugging Face & Tải mô hình lên 2x T4 GPU

In [ ]:
import sys
import os

# Thêm đường dẫn module của ReAgent-V vào sys.path
module_path = "/kaggle/working/ReAgent-V-repo/ReAgent-V"
if module_path not in sys.path:
    sys.path.append(module_path)
os.chdir(module_path)

# Cấu hình đường dẫn model tải trực tiếp từ Hugging Face
path_dict = {
    "clip_model_path": "openai/clip-vit-large-patch14-336",
    "clip_cache_dir": "/kaggle/working/cache_models",
    "whisper_model_path": "openai/whisper-large",
    "whisper_cache_dir": "/kaggle/working/cache_models",
    "llava_model_path": "lmms-lab/LLaVA-Video-7B-Qwen2",
    "llava_cache_dir": "/kaggle/working/cache_models",
}

print("⏳ Đang tải các mô hình (LLaVA-Video-7B, CLIP, Whisper)... Quá trình này có thể mất 3-5 phút.")
from ReAgentV import ReAgentV

# Tải mô hình - Accelerate sẽ tự động phân bổ đều lên 2 GPU T4 (device_map='auto')
qa_system = ReAgentV.load_default(path_dict)
print("✅ Khởi tạo hệ thống Multi-Agent thành công trên 2x GPU T4!")

## 🚀 Bước 4: Chạy Pipeline suy luận & Đánh giá Multi-Agent (ReAgent-V)

In [ ]:
import json
from ReAgentV_utils.model_inference.model_inference import llava_inference
from ReAgentV_utils.prompt_builder.prompt import tool_retrieval_prompt_template

# -----------------------------------------------------
# ĐẶT CÂU HỎI VỀ VIDEO TẠI ĐÂY:
question = "Describe the main event and action happening in this video."
# -----------------------------------------------------

print(f"\n[1/5] Lấy mẫu khung hình thông minh (ECRS Sampling) cho câu hỏi: '{question}'...")
frames, key_frames, key_indices, max_frames_num, raw_video, video_tensor = (
    qa_system.load_and_sample_video(
        question=question,
        video_path=video_path
    )
)
print(f"Đã trích xuất {len(key_frames)} key-frames quan trọng nhất.")

print("\n[2/5] Kích hoạt công cụ thị giác/âm thanh đa thể thức (OCR, ASR, Detection...)... ")
modal_info, det_top_idx, USE_OCR, USE_ASR, USE_DET = qa_system.retrieve_modal_info(
    video_path=video_path,
    question=question,
    frames=key_frames,
    raw_video=raw_video,
    clip_model=qa_system.clip_model,
    clip_processor=qa_system.clip_processor,
)

print("\n[3/5] Xây dựng Prompt & Suy luận ban đầu (Initial Answer)...")
qs = qa_system.build_multimodal_prompt(
    question=question,
    modal_info=modal_info,
    det_top_idx=det_top_idx,
    max_frames_num=max_frames_num,
    USE_DET=USE_DET,
    USE_ASR=USE_ASR,
    USE_OCR=USE_OCR,
)
initial_answer = llava_inference(qs, video_tensor)
print("--- INITIAL ANSWER ---")
print(initial_answer)

print("\n[4/5] Tác nhân phản biện (Critic Agent) kiểm tra điểm mù và sinh báo cáo đánh giá...")
critique_questions = qa_system.generate_critical_questions(
    question, initial_answer, modal_info, video_tensor
)

updated_infos = {}
for cq in critique_questions:
    tool_selection_prompt = tool_retrieval_prompt_template.format(question=cq)
    response_list = llava_inference(tool_selection_prompt, video=None)
    new_modal_info, new_det_top_idx, _, _, _ = qa_system.retrieve_modal_info(
        video_path=video_path,
        question=cq,
        frames=key_frames,
        raw_video=raw_video,
        clip_model=qa_system.clip_model,
        clip_processor=qa_system.clip_processor,
    )
    updated_infos[cq] = new_modal_info

context_infos = {question: modal_info, **updated_infos}
context_str = json.dumps(context_infos[question], indent=2)

eval_report = qa_system.generate_eval_report(
    question=question,
    context_info=context_str,
    initial_answer=initial_answer,
    video=video_tensor
)
print("--- EVALUATION & REWARD REPORT ---")
print(eval_report)

print("\n[5/5] Phản biện đa góc nhìn (Conservative, Neutral, Aggressive) -> Final Answer...")
final_answer = qa_system.get_reflective_final_answer(
    question, initial_answer, eval_report, video_tensor
)

print("\n======================================================")
print("🎉 KẾT QUẢ CUỐI CÙNG (REFLECTIVE FINAL ANSWER):")
print("======================================================")
print(final_answer)